Analysis of the Olist Brazilian e-commerce dataset to answer three
business questions related to e-commerce performance and customer experience.

**Question 1: Are repeat customers more valuable than one-time buyers?**

KPI: Number of customers in each group

KPI: Average total spend per customer, split by repeat vs. one-time buyers

**Question 2: Does delivery time affect customer satisfaction?**

KPI: Number of orders per review score

KPI: Average delivery time, grouped by review score

**Question 3: Which product categories generate the most revenue?**

KPI: Number of orders per category

KPI: Total revenue of each category


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from sqlalchemy import create_engine
import pandas as pd

DB_PATH = '/content/drive/MyDrive/olist_project/olist.db'
engine = create_engine(f'sqlite:///{DB_PATH}')

**1. How many customers are repeat buyers or one-time buyers?**


In [5]:
q1 = """
SELECT
    CASE WHEN order_count > 1 THEN 'Repeat' ELSE 'One-time' END AS customer_type,
    COUNT(*) AS num_customers,
    ROUND(AVG(total_spend), 2) AS avg_total_spend
FROM (
    SELECT c.customer_unique_id,
           COUNT(o.order_id) AS order_count,
           SUM(p.payment_value) AS total_spend
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN payments p ON o.order_id = p.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id
)
GROUP BY customer_type;
"""
result_q1 = pd.read_sql_query(q1, engine)
print(result_q1)

  customer_type  num_customers  avg_total_spend
0      One-time          87893           160.58
1        Repeat           5464           239.46


The number of total customers is 93357.
Out of that, only 5464 customers are repeat buyers and they spend $239.46 on average.

Meanwhile, one time buyers spend $160.58 on average.

Repeat buyers spend about 49% more than one-time buyers on average.

This business depends heavily on one-time purchases. So, even a small increase in repeat-purchase rate could grow revenue. This can be done through retention efforts like email follow-ups and loyalty incentives.

**2. Comparing delivery time with review score to analyze customer satisfaction**

In [6]:
q2 = """
SELECT
    r.review_score,
    ROUND(AVG(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp)), 1) AS avg_delivery_days,
    COUNT(*) AS num_orders
FROM orders o
JOIN reviews r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
GROUP BY r.review_score
ORDER BY r.review_score;
"""
result_q2 = pd.read_sql_query(q2, engine)
print(result_q2)


   review_score  avg_delivery_days  num_orders
0             1               21.3        9405
1             2               16.7        2941
2             3               14.3        7961
3             4               12.3       18987
4             5               10.7       57059


There is an inverse relationship between delivery time and review score.
The orders with a 5-star review took an average of 10.7 days to deliver, while
1-star reviews took nearly twice as long at 21.3 days.
Delivery time seems to impact customer satisfaction a lot.
Investment in faster and reliable shipping could directly improve review scores and customer retention.

**3.Which product categories generate the most revenue?**

In [9]:
q3 = """
SELECT
    ct.product_category_name_english AS category,
    SUM(oi.price) AS revenue,
    COUNT(DISTINCT oi.order_id) AS num_orders
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
JOIN products p ON oi.product_id = p.product_id
JOIN category_translation ct ON p.product_category_name = ct.product_category_name
WHERE o.order_status = 'delivered'
GROUP BY category
ORDER BY revenue DESC
LIMIT 10;
"""
result_q3 = pd.read_sql_query(q3, engine)
print(result_q3)

                category     revenue  num_orders
0          health_beauty  1233131.72        8647
1          watches_gifts  1166176.98        5495
2         bed_bath_table  1023434.76        9272
3         sports_leisure   954852.55        7530
4  computers_accessories   888724.61        6530
5        furniture_decor   711927.69        6307
6             housewares   615628.69        5743
7             cool_stuff   610204.10        3559
8                   auto   578966.65        3810
9                   toys   471286.48        3804


Health & beauty is the top revenue category ( $1.23M).

It's ahead of watches/gifts ($1.17M), despite watches/gifts having far less orders.This means it has a higher average order value per purchase.

Bed/bath/table has the highest number of orders(9,272) but lower revenue than health & beauty, implying lower-priced,higher-volume items.

This suggests different category strategies:
health & beauty may benefit from volume-driven promotions, while
watches/gifts could be leaned into as a premium, higher-margin category.